<a href="https://colab.research.google.com/github/GIND123/Inference-Guard/blob/27-modularize-finetune-harness/InferenceGuard_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


%cd /content/drive/MyDrive

Mounted at /content/drive
/content/drive/MyDrive


In [36]:
!rm -rf /content/drive/MyDrive/Inference-Guard

In [37]:
%cd /content/drive/MyDrive


!git clone -b 27-modularize-finetune-harness https://github.com/GIND123/Inference-Guard.git


%cd Inference-Guard

/content/drive/MyDrive
Cloning into 'Inference-Guard'...
remote: Enumerating objects: 218, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 218 (delta 10), reused 15 (delta 7), pack-reused 192 (from 1)
Receiving objects: 100% (218/218), 945.83 KiB | 14.78 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/drive/MyDrive/Inference-Guard


In [11]:
!ls

artifacts  data  img				      README.md  scripts  tests
configs    docs  InferenceGuard_Implementation.ipynb  reports	 src	  web


In [12]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio pytest presidio-analyzer presidio-anonymizer transformers huggingface_hub torch

In [19]:
!python tests/run_all_tests.py

InferenceGuard Master Test Suite
Project root: /content/drive/MyDrive/Inference-Guard

[Step 1/1] Running Full Test Suite via pytest...
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/Inference-Guard
plugins: typeguard-4.6.0, langsmith-0.12.1, anyio-4.14.2
collected 114 items                                                            

tests/test_calibration.py::test_compute_ece_metrics PASSED               [  0%]
tests/test_calibration.py::test_temperature_scaler_fit_synthetic PASSED  [  1%]
tests/test_calibration.py::test_temperature_scaler_reduces_ece PASSED    [  2%]
tests/test_calibration.py::test_platt_scaling_fallback_trigger PASSED    [  3%]
tests/test_calibration.py::test_calibration_save_and_load_roundtrip PASSED [  4%]
tests/test_calibration.py::test_generate_reliability_diagram_headless PASSED [  5%]
tests/

### Dataset

In [38]:
import os
import json
from datasets import load_dataset

os.makedirs("data/raw", exist_ok=True)

# Download the SynthPAI dataset

dataset = load_dataset("RobinSta/SynthPAI", split="train")


with open("data/raw/synthpai.jsonl", "w", encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps(row) + "\n")


### Finetune ModernBERT

In [39]:
from src.risk_model.train import train_risk_model

metrics = train_risk_model(
    data_path="data/raw/synthpai.jsonl",
    splits_path="artifacts/profile_splits.json",
    output_dir="artifacts/risk_model",
    epochs=3,
    batch_size=32
)
print(metrics)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'best_macro_f1': 0.46132528346960455, 'pos_weights': {'age': 14.121387283236995, 'location': 20.355102040816327, 'occupation': 3.396638655462185, 'education': 8.98473282442748}, 'history': [{'val_loss': 3.3908721571383267, 'per_attribute': {'age': {'f1': 0.19298245614035087, 'pr_auc': 0.15462307974290168, 'roc_auc': 0.7203065826697638, 'balanced_accuracy': 0.6692847910772874, 'pos_count': 83}, 'location': {'f1': 0.30036630036630035, 'pr_auc': 0.3924330663486832, 'roc_auc': 0.8723608193277311, 'balanced_accuracy': 0.8013655462184874, 'pos_count': 56}, 'occupation': {'f1': 0.5811732605729877, 'pr_auc': 0.5872461143885259, 'roc_auc': 0.852860300688228, 'balanced_accuracy': 0.7737172151065264, 'pos_count': 281}, 'education': {'f1': 0.4560669456066946, 'pr_auc': 0.45926933687209864, 'roc_auc': 0.8598748786538669, 'balanced_accuracy': 0.7854923956423255, 'pos_count': 146}}, 'macro_f1': 0.3826472406715834, 'macro_pr_auc': 0.39839289933805233, 'macro_roc_auc': 0.8263506453348975, 'macro_balan

### Qwen SFT Dataset

In [44]:
import importlib
import src.rewriter.generate_training_data

importlib.reload(src.rewriter.generate_training_data)
from src.rewriter.generate_training_data import generate_sft_dataset


sft_metrics = generate_sft_dataset(
    synthpai_path="data/raw/synthpai.jsonl",
    splits_path="artifacts/profile_splits.json",
    output_chatml_path="artifacts/rewriter_sft_chatml.jsonl",
    output_alpaca_path="artifacts/rewriter_sft_alpaca.json",
    max_risk_threshold=0.30,
    min_cosine_threshold=0.50
)
print(sft_metrics)

{'total_source_records': 7823, 'train_records_retained': 5232, 'rejected_non_train_profiles': 2591, 'total_candidates_evaluated': 5244, 'accepted_samples': 27, 'output_chatml_file': 'artifacts/rewriter_sft_chatml.jsonl', 'output_alpaca_file': 'artifacts/rewriter_sft_alpaca.json'}


In [51]:

!pkill -f uvicorn

!git pull origin 27-modularize-finetune-harness


import subprocess
import time
import urllib.request

get_ipython().system_raw('uvicorn web.api:app --host 0.0.0.0 --port 8000 &')
time.sleep(6)

# colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
proc = subprocess.Popen(['npx', 'localtunnel', '--port', '8000'], stdout=subprocess.PIPE, text=True)
url = proc.stdout.readline().strip().replace('url: ', '')


print(f"INFERENCEGUARD UI IS LIVE AT: {url}")


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 862 bytes | 19.00 KiB/s, done.
From https://github.com/GIND123/Inference-Guard
 * branch            27-modularize-finetune-harness -> FETCH_HEAD
   05b37f3..2f454d1  27-modularize-finetune-harness -> origin/27-modularize-finetune-harness
Updating 05b37f3..2f454d1
Fast-forward
 web/index.html | 14 ++++++++------
 1 file changed, 8 insertions(+), 6 deletions(-)
INFERENCEGUARD UI IS LIVE AT: your url is: https://ten-eyes-stay.loca.lt
